In [93]:
import os
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as pl
import shutil
import nrrd
import cv2
import SimpleITK as sitk
import ast
import matplotlib.pyplot as plt


coord_df = pd.read_excel('/Users/Francisco/Desktop/RestrepoProjects/Active/Regeneron retrospective radiomics/databases/segmentations/mrns_valentin_segmented_cases.xlsx')
first_row_file = 2386
last_row_file = 2401
file_rows = list(range(first_row_file-2,last_row_file-1))
rows = coord_df.iloc[file_rows[0]]['rows'].astype(int)
cols = coord_df.iloc[file_rows[0]]['cols'].astype(int)
slices = coord_df.iloc[file_rows[0]]['slices'].astype(int)
img_size = (slices,rows,cols)
x_spacing = coord_df.iloc[file_rows[0]]['x_spacing']
y_spacing = coord_df.iloc[file_rows[0]]['y_spacing']
z_spacing = coord_df.iloc[file_rows[0]]['z_spacing']
x_origin = coord_df.iloc[file_rows[0]]['x_origin']
y_origin = coord_df.iloc[file_rows[0]]['y_origin']
z_origin = coord_df.iloc[file_rows[0]]['z_origin']

matrix = np.zeros(img_size, dtype=np.uint8)


for row in file_rows:
    ind = row-first_row_file+2
    slice_no = coord_df.iloc[row][4]
    print(slice_no)
    seg_coords = coord_df.iloc[row][14]
    pts = ast.literal_eval(seg_coords)
    x_coords = pts[0::2]
    y_coords = pts[1::2]
    pts = np.array([x_coords, y_coords], dtype=np.int32).T
    mask = np.zeros((rows,cols))
    matrix[slices - slice_no-1,:,:] = cv2.fillPoly(mask, np.array([pts], dtype=np.int32), 1)
    
# May need to do this depending on how data were collected    
#matrix=matrix[::-1, :, :]

seg = sitk.GetImageFromArray(matrix)
seg.SetSpacing((x_spacing, y_spacing, z_spacing))  # Set spacing as needed
seg.SetOrigin((x_origin, y_origin, z_origin)) 
seg.SetMetaData("Segmentation_ContainedRepresentationNames", "Binary labelmap")
seg.SetMetaData("Segmentation_MasterRepresentation", "Binary labelmap")
seg.SetMetaData("Segment0_ID", "Segment_1")
seg.SetMetaData("Segment0_Name", "MySegment")
seg.SetMetaData("Segment0_Color", "0.0 1.0 0.0")
seg.SetMetaData("Segment0_LabelValue", "1")
mask_nrrd_path = '/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/Inter-reader_segmentations/Amine/CE2-2301-153/P153_t2.seg.nrrd'
sitk.WriteImage(seg, mask_nrrd_path)


def get_nrrd_info(file_path):
    header = nrrd.read_header(file_path)
    
    dims = header.get('sizes')

    # 1. Get Origin
    origin = header.get('space origin')
    
    # 2. Get Spacing from Space Directions
    directions = header.get('space directions')
    if directions is not None:
        # Calculate magnitude of each vector in the 3x3 matrix
        spacing = [np.linalg.norm(v) for v in directions if v is not None]
    else:
        spacing = header.get('spacings') # Fallback for simple headers

    return origin, spacing

24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39


/var/folders/t2/wb_946453g5_cv_c36m_1kvh0000gq/T/ipykernel_1794/1869283356.py:34: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slice_no = coord_df.iloc[row][4]
/var/folders/t2/wb_946453g5_cv_c36m_1kvh0000gq/T/ipykernel_1794/1869283356.py:36: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  seg_coords = coord_df.iloc[row][14]


In [90]:
import nrrd
import numpy as np
def get_nrrd_info(file_path):
    header = nrrd.read_header(file_path)
    
    dims = header.get('sizes')
    # 1. Get Origin
    origin = header.get('space origin')
    
    # 2. Get Spacing from Space Directions
    directions = header.get('space directions')
    if directions is not None:
        # Calculate magnitude of each vector in the 3x3 matrix
        spacing = [np.linalg.norm(v) for v in directions if v is not None]
    else:
        spacing = header.get('spacings') # Fallback for simple headers

    return origin, spacing, dims

# Usage
origin, spacing, dims = get_nrrd_info('/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/Inter-reader_segmentations/Amine/CE2-2301-153/P153_t2_N4.nrrd')
print(f"Origin:  {origin}")
print(f"Spacing: {spacing}")
print(f"Dimensions: {dims} (X, Y, Z)")

origin[0]

Origin:  [-163.67429543 -104.58791542  -89.38733673]
Spacing: [1.3671875, 1.3671875, 6.499999999999906]
Dimensions: [256 192  33] (X, Y, Z)


-163.67429542542

In [87]:
img_size



(-9223372036854775808, -9223372036854775808, -9223372036854775808)